# LangGraph Data Cleaning Workflow with Human-in-the-Loop

## 1. Project Overview

**Task:** Build a **LangGraph state graph** that profiles a dataset, suggests cleaning actions, pauses for human approval, and applies the approved transformations — all as an inspectable, resumable pipeline.

**Why this matters:** Data cleaning is 60-80% of a data scientist's work, but it's dangerous to automate blindly. Dropping a column, imputing missing values, or removing outliers can silently destroy signal. A graph-based workflow lets an LLM propose intelligent cleaning steps while keeping a human in the approval loop.

**Pipeline Nodes:**

```
  START
    │
    ▼
  ┌──────────────────────┐
  │  profile_data         │  compute stats, detect types, find issues
  └──────────┬───────────┘
             │
             ▼
  ┌──────────────────────┐
  │  suggest_cleaning     │  propose specific transformations
  └──────────┬───────────┘
             │
             ▼
  ┌──────────────────────┐
  │  approval_gate        │  ★ PAUSE — human reviews each action
  └──────────┬───────────┘
        conditional
       ┌────┼──────┐
       ▼    ▼      ▼
    approve revise reject
       │    │      │
       │    └→ back to suggest_cleaning
       │           │
       ▼           ▼
  ┌──────────────────────┐
  │  apply_transforms     │  execute approved cleaning steps
  └──────────┬───────────┘
             │
             ▼
  ┌──────────────────────┐
  │  compile_report       │  before/after comparison + audit log
  └──────────┬───────────┘
             │
             ▼
           END
```

**Stack:** `LangGraph`, `ChatOllama` + `qwen3.5:9b`, `pandas`

## 2. Learning Goals

| # | Skill |
|---|-------|
| 1 | Design a **TypedDict state schema** for a data-cleaning pipeline |
| 2 | Build **single-responsibility nodes** — profile, suggest, approve, apply, report |
| 3 | Implement a **human-in-the-loop approval gate** that pauses the graph |
| 4 | Use **conditional edges** to branch on human decisions |
| 5 | Apply **partial state updates** — each node returns only what it changed |
| 6 | See **before/after metrics** proving the cleaning worked |
| 7 | Understand **why human-in-the-loop matters** for data integrity |

## 3. Why Human-in-the-Loop for Data Cleaning?

### The problem with fully automated cleaning
```
  AUTO MODE:  df.dropna()  →  silently drops 30% of rows containing valid partial data
  AUTO MODE:  df.drop_duplicates()  →  removes legitimate repeat measurements
  AUTO MODE:  outlier removal  →  deletes the rare events you actually want to study
```

### The human-in-the-loop approach
```
  1. LLM PROPOSES:  "Column 'age' has 12% nulls → impute with median (34)"
  2. HUMAN REVIEWS:  "Actually, nulls in 'age' mean 'not disclosed' — keep as-is"
  3. LLM REVISES:   Updated proposal skips 'age' imputation
  4. HUMAN APPROVES: Final plan applied with full audit trail
```

### When to use each pattern

| Pattern | Use When | Risk |
|---------|----------|------|
| Fully automated | Standardized ETL on well-known schemas | Low |
| Human-in-the-loop | Exploratory analysis, new datasets, critical decisions | Medium |
| Fully manual | Regulated data (HIPAA, financial audits) | Very low but slow |

This notebook implements the **middle path** — LLM intelligence with human guardrails.

## 4. State Passing in This Pipeline

```
  ┌──────────────────────────────────────────────────────────────────┐
  │               HOW STATE ACCUMULATES                              │
  ├──────────────────────────────────────────────────────────────────┤
  │                                                                  │
  │  After profile_data:                                            │
  │    state += {profile_report: "...stats..."}                      │
  │                                                                  │
  │  After suggest_cleaning:                                        │
  │    state += {cleaning_plan: "1. Drop col X..."}                  │
  │    → profile_report is STILL there, untouched                   │
  │                                                                  │
  │  After approval_gate:                                           │
  │    state += {decision: "approve", approval_notes: "..."}         │
  │    → profile_report AND cleaning_plan both preserved            │
  │                                                                  │
  │  After apply_transforms:                                        │
  │    state += {cleaned_df_json: "...", transform_log: "..."}       │
  │    → the ORIGINAL df_json is still in state for comparison      │
  │                                                                  │
  │  KEY: Nodes only return their OWN outputs.                      │
  │  LangGraph merges them into the full state.                      │
  └──────────────────────────────────────────────────────────────────┘
```

## 5. Environment Setup

In [ ]:
# Uncomment if any package is missing
# !pip install -q langchain langchain-ollama langchain-core langgraph pandas numpy

print("Dependencies: langchain, langgraph, pandas, numpy")
print("LLM: Ollama with qwen3.5:9b (must be running locally)")

## 6. Imports & Configuration

In [ ]:
import os
import re
import json
import textwrap
import warnings
from typing import TypedDict

import numpy as np
import pandas as pd

os.environ["USE_TF"] = "0"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
warnings.filterwarnings("ignore")

from langchain_ollama import ChatOllama
from langchain_core.messages import HumanMessage, SystemMessage
from langgraph.graph import StateGraph, START, END

LLM_MODEL = "qwen3.5:9b"

print("All imports OK")
print(f"LLM: {LLM_MODEL}")
print(f"pandas {pd.__version__}, numpy {np.__version__}")

## 7. Helper Functions

In [ ]:
def clean(text: str) -> str:
    if "<think>" in text:
        text = text.split("</think>")[-1].strip()
    return text.strip()


def parse_json(text: str):
    text = clean(text)
    if "```" in text:
        text = re.sub(r"```(?:json)?\s*", "", text)
        text = text.replace("```", "")
    start = text.find("{")
    alt = text.find("[")
    if alt >= 0 and (start < 0 or alt < start):
        start = alt
    end = max(text.rfind("}"), text.rfind("]")) + 1
    if start >= 0 and end > start:
        try:
            return json.loads(text[start:end])
        except json.JSONDecodeError:
            pass
    return None


def ask(prompt: str, system: str = "", temperature: float = 0.2) -> str:
    llm = ChatOllama(model=LLM_MODEL, temperature=temperature)
    messages = []
    if system:
        messages.append(SystemMessage(content=system))
    messages.append(HumanMessage(content=prompt))
    response = llm.invoke(messages)
    return clean(response.content)


def wrap_print(text: str, width: int = 100):
    for line in text.split("\n"):
        if line.strip():
            print(textwrap.fill(line, width=width))
        else:
            print()


# Quick LLM check
test = ask("Say 'ready'.")
print(f"LLM check: {test[:30]}")

---

# Part A — Synthetic Dataset & State Schema

## 8. Create a Realistic Messy Dataset

We'll build a dataset with real-world data quality issues:
- Missing values (random and systematic)
- Inconsistent formatting (mixed case, trailing spaces)
- Outliers
- Duplicate rows
- Wrong data types (numbers stored as strings)

In [ ]:
np.random.seed(42)
n = 200

# Base clean data
names = np.random.choice(
    ["Alice Johnson", "Bob Smith", "Carol White", "David Brown",
     "Eve Davis", "Frank Wilson", "Grace Lee", "Henry Taylor",
     "Ivy Martinez", "Jack Anderson"], n
)
ages = np.random.randint(22, 65, n).astype(float)
salaries = np.round(np.random.normal(65000, 15000, n), 2)
departments = np.random.choice(
    ["Engineering", "Marketing", "Sales", "HR", "Finance"], n
)
ratings = np.round(np.random.uniform(1.0, 5.0, n), 1)
join_dates = pd.date_range("2018-01-01", periods=n, freq="5D").strftime("%Y-%m-%d").tolist()
emails = [f"{name.split()[0].lower()}.{name.split()[1].lower()}@company.com" for name in names]

df = pd.DataFrame({
    "employee_name": names,
    "age": ages,
    "salary": salaries,
    "department": departments,
    "performance_rating": ratings,
    "join_date": join_dates,
    "email": emails,
})

# --- Inject data quality issues ---

# 1. Missing values (random)
for col in ["age", "salary", "performance_rating"]:
    mask = np.random.random(n) < 0.08
    df.loc[mask, col] = np.nan

# 2. Missing values (systematic — new hires have no rating)
recent = df["join_date"] > "2020-06-01"
df.loc[recent & (np.random.random(n) < 0.4), "performance_rating"] = np.nan

# 3. Inconsistent department names
noise_idx = np.random.choice(n, 25, replace=False)
dept_variants = {"Engineering": ["engineering", "ENGINEERING", "Eng", "engineering "],
                 "Marketing": ["marketing", "MARKETING", "Mktg"],
                 "Sales": ["sales", "SALES", "sales "],
                 "HR": ["hr", "Human Resources", "H.R."],
                 "Finance": ["finance", "FINANCE", "Fin"]}
for i in noise_idx:
    orig = df.loc[i, "department"]
    if orig in dept_variants:
        df.loc[i, "department"] = np.random.choice(dept_variants[orig])

# 4. Outliers in salary
outlier_idx = np.random.choice(n, 5, replace=False)
df.loc[outlier_idx, "salary"] = np.random.choice([250000, 500000, -5000, 0, 999999], 5)

# 5. Duplicate rows
dup_idx = np.random.choice(n, 8, replace=False)
df = pd.concat([df, df.iloc[dup_idx]], ignore_index=True)

# 6. Salary as string in some rows
str_idx = np.random.choice(len(df), 10, replace=False)
df["salary"] = df["salary"].astype(object)
for i in str_idx:
    if pd.notna(df.loc[i, "salary"]):
        df.loc[i, "salary"] = f"${df.loc[i, 'salary']:,.2f}"

# 7. Trailing whitespace in names
ws_idx = np.random.choice(len(df), 15, replace=False)
for i in ws_idx:
    df.loc[i, "employee_name"] = df.loc[i, "employee_name"] + "  "

print(f"Dataset created: {df.shape[0]} rows × {df.shape[1]} columns")
print(f"\nInjected issues:")
print(f"  Missing values:       {df.isna().sum().sum()} total nulls")
print(f"  Unique departments:   {df['department'].nunique()} (should be 5)")
print(f"  Duplicate rows:       {df.duplicated().sum()}")
print(f"  Salary dtype:         {df['salary'].dtype} (mixed numeric+string)")
print(f"\nSample rows:")
df.head(10)

## 9. State Schema

The state carries the raw data, the profile, proposed cleaning plan, human decision, cleaned data, and an audit log.

**Field ownership:**

| Field | Written By | Read By |
|-------|-----------|--------|
| `df_json` | Input | profile_data, apply_transforms |
| `profile_report` | profile_data | suggest_cleaning |
| `cleaning_plan` | suggest_cleaning | approval_gate |
| `decision` | approval_gate | routing function |
| `approval_notes` | approval_gate | compile_report |
| `revision_feedback` | approval_gate | suggest_cleaning (on revision) |
| `revision_count` | approval_gate | routing function |
| `cleaned_df_json` | apply_transforms | compile_report |
| `transform_log` | apply_transforms | compile_report |
| `final_report` | compile_report | caller |
| `current_node` | every node | debugging |

In [ ]:
class CleaningState(TypedDict):
    """State schema for the data cleaning pipeline.

    Design notes:
    - df_json: original DataFrame as JSON (for reproducibility)
    - profile_report: stats on nulls, types, outliers, duplicates
    - cleaning_plan: proposed transformations from LLM
    - decision: human verdict — approve / revise / reject
    - revision_feedback: human's improvement instructions
    - revision_count: safety cap for revision loops
    - cleaned_df_json: the cleaned DataFrame as JSON
    - transform_log: record of every transformation applied
    - final_report: before/after comparison with audit trail
    """
    df_json: str
    profile_report: str
    cleaning_plan: str
    decision: str
    approval_notes: str
    revision_feedback: str
    revision_count: int
    cleaned_df_json: str
    transform_log: str
    final_report: str
    current_node: str


print("State schema: CleaningState")
for name, typ in CleaningState.__annotations__.items():
    print(f"  {name}: {typ}")

---

# Part B — Build the Graph Nodes

## 10. Node 1: Profile Data

Compute concrete statistics and detect issues. This is **deterministic** — no LLM needed. The profile feeds into the LLM-powered suggestion node.

**Why separate profiling from suggestion?** Profiling is factual (row count, null percentage). Suggestions are judgmental ("drop this column"). Mixing them makes the LLM's reasoning harder to audit.

In [ ]:
def profile_data(state: CleaningState) -> dict:
    """Node: Compute data quality profile (deterministic, no LLM)."""
    print("  [NODE] profile_data")
    df = pd.read_json(state["df_json"], orient="records")

    lines = []
    lines.append(f"DATASET PROFILE")
    lines.append(f"  Rows: {len(df)}")
    lines.append(f"  Columns: {len(df.columns)}")
    lines.append(f"  Column names: {list(df.columns)}")
    lines.append("")

    # Column-level stats
    lines.append("COLUMN DETAILS:")
    for col in df.columns:
        dtype = str(df[col].dtype)
        nulls = df[col].isna().sum()
        null_pct = round(100 * nulls / len(df), 1)
        unique = df[col].nunique()
        lines.append(f"  {col}:")
        lines.append(f"    dtype={dtype}, nulls={nulls} ({null_pct}%), unique={unique}")
        if dtype in ["float64", "int64"]:
            lines.append(f"    min={df[col].min()}, max={df[col].max()}, "
                         f"mean={df[col].mean():.1f}, std={df[col].std():.1f}")
        elif dtype == "object":
            sample_vals = df[col].dropna().head(5).tolist()
            lines.append(f"    samples: {sample_vals}")
            # Check for whitespace issues
            ws = df[col].dropna().apply(lambda x: x != x.strip() if isinstance(x, str) else False).sum()
            if ws > 0:
                lines.append(f"    ⚠ whitespace issues: {ws} values have leading/trailing spaces")

    # Duplicates
    dup_count = df.duplicated().sum()
    lines.append(f"\nDUPLICATES: {dup_count} duplicate rows ({round(100*dup_count/len(df), 1)}%)")

    # Mixed types detection
    lines.append("\nMIXED TYPE DETECTION:")
    for col in df.columns:
        types_found = df[col].dropna().apply(type).value_counts().to_dict()
        type_names = {str(k.__name__): v for k, v in types_found.items()}
        if len(type_names) > 1:
            lines.append(f"  ⚠ {col}: mixed types — {type_names}")

    # Inconsistent categorical values
    lines.append("\nCATEGORICAL CONSISTENCY:")
    for col in df.select_dtypes(include="object").columns:
        vals = df[col].dropna().unique()
        if 3 < len(vals) < 30:
            stripped = df[col].dropna().str.strip().str.lower().nunique()
            if stripped < len(vals):
                lines.append(f"  ⚠ {col}: {len(vals)} raw values collapse to {stripped} after normalization")
                lines.append(f"    raw values: {sorted(vals)[:15]}")

    # Outlier detection (IQR method)
    lines.append("\nOUTLIER DETECTION (IQR):")
    for col in df.select_dtypes(include=["float64", "int64"]).columns:
        q1 = df[col].quantile(0.25)
        q3 = df[col].quantile(0.75)
        iqr = q3 - q1
        lower = q1 - 1.5 * iqr
        upper = q3 + 1.5 * iqr
        outliers = ((df[col] < lower) | (df[col] > upper)).sum()
        if outliers > 0:
            out_vals = df[(df[col] < lower) | (df[col] > upper)][col].tolist()
            lines.append(f"  ⚠ {col}: {outliers} outliers (range [{lower:.1f}, {upper:.1f}])")
            lines.append(f"    outlier values: {out_vals[:10]}")

    report = "\n".join(lines)
    print(f"    Profile: {len(report)} chars, {len(lines)} lines")
    return {"profile_report": report, "current_node": "profile_data"}


print("Node defined: profile_data")
print("  READS:  df_json")
print("  WRITES: profile_report")
print("  NOTE:   No LLM — purely deterministic statistics")

## 11. Node 2: Suggest Cleaning Actions

The LLM reads the profile report and proposes specific, numbered cleaning actions. If this is a revision round, it also considers human feedback.

In [ ]:
SUGGEST_SYSTEM = """You are a data engineering expert. Given a dataset profile,
propose a numbered list of cleaning actions. For each action:

- ACTION NUMBER and DESCRIPTION
- COLUMN(S) affected
- RISK: low / medium / high
- RATIONALE: why this cleaning is needed
- METHOD: the specific pandas operation (e.g., df['col'].fillna(median), df.drop_duplicates())

Order by priority (most impactful first). Propose 4-8 actions.
NEVER propose dropping entire columns unless truly useless.
NEVER propose removing more than 5% of rows without strong justification."""

SUGGEST_PROMPT = """DATASET PROFILE:
{profile}

{revision_section}

Propose 4-8 cleaning actions with method, risk, and rationale for each."""


def suggest_cleaning(state: CleaningState) -> dict:
    """Node: Propose data cleaning actions based on the profile."""
    print("  [NODE] suggest_cleaning")

    revision_section = ""
    if state.get("revision_feedback") and state.get("revision_count", 0) > 0:
        revision_section = (
            f"REVIEWER FEEDBACK (incorporate these changes):\n"
            f"{state['revision_feedback']}\n\n"
            f"PREVIOUS PLAN (improve, don't restart):\n"
            f"{state['cleaning_plan'][:800]}"
        )
        print(f"    Revision round {state['revision_count']} — incorporating feedback")

    plan = ask(
        SUGGEST_PROMPT.format(
            profile=state["profile_report"][:3000],
            revision_section=revision_section,
        ),
        system=SUGGEST_SYSTEM,
        temperature=0.2,
    )
    print(f"    Plan generated: {len(plan)} chars")
    return {"cleaning_plan": plan, "current_node": "suggest_cleaning"}


print("Node defined: suggest_cleaning")
print("  READS:  profile_report, revision_feedback (if revision)")
print("  WRITES: cleaning_plan")

## 12. Node 3: Approval Gate

### The Human-in-the-Loop Design

```
  ┌─────────────────────────────────────────────────────────────┐
  │                APPROVAL GATE DESIGN                         │
  ├─────────────────────────────────────────────────────────────┤
  │                                                             │
  │  DISPLAY:                                                   │
  │    • Dataset profile summary (row/col/null counts)         │
  │    • Each proposed cleaning action with rationale          │
  │    • Risk level for each action                            │
  │                                                             │
  │  HUMAN OPTIONS:                                            │
  │    approve — execute all proposed actions                   │
  │    revise  — send feedback, get revised proposal            │
  │    reject  — skip cleaning, go straight to report          │
  │                                                             │
  │  WHY THIS MATTERS:                                         │
  │    • Prevents destructive cleaning (dropping valid data)   │
  │    • Human catches domain-specific nuance                  │
  │    • Creates auditable decision trail                      │
  │    • Revision loop improves plan without starting over     │
  │                                                             │
  │  PRODUCTION IMPLEMENTATIONS:                               │
  │    • input() for interactive notebooks                     │
  │    • LangGraph interrupt() + checkpointing                 │
  │    • Slack bot with reaction-based approval                │
  │    • Web form via webhook callback                         │
  └─────────────────────────────────────────────────────────────┘
```

In [ ]:
# Pre-defined human decisions for reproducibility
# In production, replace with input() or LangGraph interrupt()
SIMULATED_REVIEWS = [
    (
        "revise",
        "Good coverage but I want explicit handling of the salary column's mixed dollar-sign strings.",
        "The salary column has values like '$65,000.00' mixed with numeric floats. "
        "Add a step to strip the dollar sign and commas, then convert to float BEFORE "
        "doing any statistical outlier detection on salary. "
        "Also, keep the negative salary (-5000) as an outlier to remove, not to keep.",
    ),
    (
        "approve",
        "Plan looks comprehensive now. Proceed with all actions.",
        "",
    ),
]


def approval_gate(state: CleaningState) -> dict:
    """Node: Human reviews the proposed cleaning plan.

    In production, this would PAUSE the graph and wait for human input.
    Here we simulate decisions for reproducibility.
    """
    print("  [NODE] approval_gate")
    rev = state.get("revision_count", 0)

    # Display what the reviewer sees
    print("  " + "=" * 55)
    print("  ★ HUMAN REVIEW CHECKPOINT ★")
    print(f"  Review round: {rev + 1}")
    print("  " + "=" * 55)
    print("  Profile highlights:")
    for line in state["profile_report"][:300].split("\n")[:6]:
        print(f"    {line}")
    print("  " + "-" * 55)
    print("  Proposed cleaning plan:")
    for line in state["cleaning_plan"][:500].split("\n")[:10]:
        print(f"    {line}")
    print("  " + "-" * 55)

    # Get simulated decision
    idx = min(rev, len(SIMULATED_REVIEWS) - 1)
    decision, notes, feedback = SIMULATED_REVIEWS[idx]

    print(f"  Decision: {decision.upper()}")
    print(f"  Notes: {notes[:80]}")
    if feedback:
        print(f"  Feedback: {feedback[:80]}...")

    return {
        "decision": decision,
        "approval_notes": notes,
        "revision_feedback": feedback,
        "revision_count": rev + 1,
        "current_node": "approval_gate",
    }


print("Node defined: approval_gate")
print("  READS:  profile_report, cleaning_plan")
print("  WRITES: decision, approval_notes, revision_feedback, revision_count")

## 13. Node 4: Apply Transformations

Executes the approved cleaning plan using deterministic pandas operations. The LLM is **not** used to execute code — it only parses the plan into actionable steps. The actual transforms are hard-coded for safety.

**Why hard-code transforms instead of using LLM-generated code?** Running arbitrary LLM-generated code on your data is dangerous. The approved plan tells us *what* to do; we use known-safe pandas operations to do it.

In [ ]:
def apply_transforms(state: CleaningState) -> dict:
    """Node: Execute approved cleaning actions with deterministic pandas ops."""
    print("  [NODE] apply_transforms")
    df = pd.read_json(state["df_json"], orient="records")
    log = []
    before_rows = len(df)
    before_nulls = df.isna().sum().sum()

    # TRANSFORM 1: Strip whitespace from string columns
    for col in df.select_dtypes(include="object").columns:
        before = df[col].apply(lambda x: x != x.strip() if isinstance(x, str) else False).sum()
        if before > 0:
            df[col] = df[col].apply(lambda x: x.strip() if isinstance(x, str) else x)
            log.append(f"STRIP WHITESPACE: {col} — fixed {before} values")

    # TRANSFORM 2: Normalize salary (strip $, commas → float)
    if "salary" in df.columns:
        def parse_salary(val):
            if isinstance(val, str):
                cleaned = val.replace("$", "").replace(",", "").strip()
                try:
                    return float(cleaned)
                except ValueError:
                    return np.nan
            return val

        before_types = df["salary"].apply(type).value_counts().to_dict()
        df["salary"] = df["salary"].apply(parse_salary).astype(float)
        log.append(f"NORMALIZE SALARY: converted mixed types {before_types} → float64")

    # TRANSFORM 3: Standardize department names
    if "department" in df.columns:
        dept_map = {
            "engineering": "Engineering", "eng": "Engineering",
            "marketing": "Marketing", "mktg": "Marketing",
            "sales": "Sales",
            "hr": "HR", "human resources": "HR", "h.r.": "HR",
            "finance": "Finance", "fin": "Finance",
        }
        before_unique = df["department"].nunique()
        df["department"] = df["department"].str.strip().str.lower().map(
            lambda x: dept_map.get(x, x.title())
        )
        after_unique = df["department"].nunique()
        log.append(f"STANDARDIZE DEPARTMENTS: {before_unique} variants → {after_unique} canonical names")

    # TRANSFORM 4: Remove salary outliers (negative or > 300K)
    if "salary" in df.columns:
        outlier_mask = (df["salary"] < 0) | (df["salary"] > 300000)
        outlier_count = outlier_mask.sum()
        if outlier_count > 0:
            outlier_vals = df.loc[outlier_mask, "salary"].tolist()
            df.loc[outlier_mask, "salary"] = np.nan
            log.append(f"REMOVE SALARY OUTLIERS: set {outlier_count} values to NaN — {outlier_vals}")

    # TRANSFORM 5: Remove exact duplicate rows
    dup_count = df.duplicated().sum()
    if dup_count > 0:
        df = df.drop_duplicates().reset_index(drop=True)
        log.append(f"REMOVE DUPLICATES: dropped {dup_count} exact duplicate rows")

    # TRANSFORM 6: Impute numeric nulls with median
    for col in ["age", "salary", "performance_rating"]:
        if col in df.columns:
            null_count = df[col].isna().sum()
            if null_count > 0:
                median_val = df[col].median()
                df[col] = df[col].fillna(median_val)
                log.append(f"IMPUTE NULLS: {col} — filled {null_count} nulls with median ({median_val:.1f})")

    after_rows = len(df)
    after_nulls = df.isna().sum().sum()
    transform_log = "\n".join(log)

    print(f"    Transforms applied: {len(log)}")
    print(f"    Rows: {before_rows} → {after_rows}")
    print(f"    Nulls: {before_nulls} → {after_nulls}")

    return {
        "cleaned_df_json": df.to_json(orient="records"),
        "transform_log": transform_log,
        "current_node": "apply_transforms",
    }


print("Node defined: apply_transforms")
print("  READS:  df_json, cleaning_plan (implicit via hard-coded steps)")
print("  WRITES: cleaned_df_json, transform_log")

## 14. Node 5: Compile Report

Produces a before/after comparison report with full audit trail.

In [ ]:
def compile_report(state: CleaningState) -> dict:
    """Node: Generate before/after comparison with audit trail."""
    print("  [NODE] compile_report")

    status = "APPROVED & APPLIED" if state["decision"] == "approve" else "REJECTED (no transforms)"

    # Load both DataFrames for comparison
    df_before = pd.read_json(state["df_json"], orient="records")
    if state["decision"] == "approve" and state["cleaned_df_json"]:
        df_after = pd.read_json(state["cleaned_df_json"], orient="records")
    else:
        df_after = df_before.copy()

    report_lines = [
        "=" * 60,
        "DATA CLEANING REPORT",
        f"Status: {status}",
        f"Revision rounds: {state['revision_count']}",
        f"Reviewer notes: {state['approval_notes']}",
        "=" * 60,
        "",
        "BEFORE / AFTER COMPARISON:",
        f"  Rows:        {len(df_before):>6}  →  {len(df_after):<6}",
        f"  Columns:     {len(df_before.columns):>6}  →  {len(df_after.columns):<6}",
        f"  Total nulls: {df_before.isna().sum().sum():>6}  →  {df_after.isna().sum().sum():<6}",
    ]

    # Column-level null comparison
    report_lines.append("\n  Column-level nulls:")
    for col in df_before.columns:
        nb = df_before[col].isna().sum()
        na = df_after[col].isna().sum() if col in df_after.columns else "—"
        marker = " ✓" if na == 0 and nb > 0 else ""
        report_lines.append(f"    {col:<25} {nb:>4}  →  {na}{marker}")

    # Categorical consistency check
    if "department" in df_after.columns:
        report_lines.append(f"\n  Departments: {sorted(df_after['department'].unique())}")

    # Duplicates
    report_lines.append(f"\n  Duplicates: {df_before.duplicated().sum()} → {df_after.duplicated().sum()}")

    # Transform audit log
    if state.get("transform_log"):
        report_lines.append("\nTRANSFORM LOG (audit trail):")
        for line in state["transform_log"].split("\n"):
            report_lines.append(f"  ✓ {line}")

    final = "\n".join(report_lines)
    print(f"    Report: {len(final)} chars")
    return {"final_report": final, "current_node": "compile_report"}


print("Node defined: compile_report")
print("  READS:  df_json, cleaned_df_json, decision, approval_notes, transform_log")
print("  WRITES: final_report")

---

# Part C — Conditional Routing & Graph Assembly

## 15. Routing After Approval Gate

Three paths:
- **approve** → `apply_transforms` → `compile_report`
- **revise** (under cap) → loop back to `suggest_cleaning`
- **revise** (at cap) or **reject** → `compile_report` directly

In [ ]:
MAX_REVISIONS = 3


def route_after_approval(state: CleaningState) -> str:
    """Conditional edge: route based on human decision."""
    decision = state.get("decision", "approve")
    revisions = state.get("revision_count", 0)

    if decision == "approve":
        print(f"    [ROUTE] approved → apply_transforms")
        return "apply_transforms"

    if decision == "reject":
        print(f"    [ROUTE] rejected → compile_report (no transforms)")
        return "compile_report"

    # revise
    if revisions >= MAX_REVISIONS:
        print(f"    [ROUTE] max revisions ({MAX_REVISIONS}) → forcing compile_report")
        return "compile_report"

    print(f"    [ROUTE] revise → suggest_cleaning (round {revisions})")
    return "suggest_cleaning"


print(f"Routing function: route_after_approval (max {MAX_REVISIONS} revisions)")
print("  approve      → apply_transforms")
print("  reject       → compile_report (skip cleaning)")
print("  revise (ok)  → suggest_cleaning (loop)")
print("  revise (cap) → compile_report")

## 16. Assemble the StateGraph

In [ ]:
workflow = StateGraph(CleaningState)

# Register nodes
workflow.add_node("profile_data", profile_data)
workflow.add_node("suggest_cleaning", suggest_cleaning)
workflow.add_node("approval_gate", approval_gate)
workflow.add_node("apply_transforms", apply_transforms)
workflow.add_node("compile_report", compile_report)

# Linear edges: START → profile → suggest → approval
workflow.add_edge(START, "profile_data")
workflow.add_edge("profile_data", "suggest_cleaning")
workflow.add_edge("suggest_cleaning", "approval_gate")

# Conditional edge after approval
workflow.add_conditional_edges(
    "approval_gate",
    route_after_approval,
    {
        "apply_transforms": "apply_transforms",
        "suggest_cleaning": "suggest_cleaning",
        "compile_report": "compile_report",
    },
)

# After transforms, always compile report
workflow.add_edge("apply_transforms", "compile_report")

# Terminal
workflow.add_edge("compile_report", END)

# Compile
app = workflow.compile()

print("Graph compiled!")
print(f"Nodes: {list(workflow.nodes.keys())}")
print()
print("Flow:")
print("  START → profile_data → suggest_cleaning → approval_gate")
print("  approval_gate ──(approve)──→ apply_transforms → compile_report → END")
print("  approval_gate ──(reject)───→ compile_report → END")
print("  approval_gate ──(revise)───→ suggest_cleaning (loop)")

In [ ]:
# Visualize the graph
try:
    mermaid_str = app.get_graph().draw_mermaid()
    print("Mermaid diagram:")
    print(mermaid_str)
except Exception as e:
    print(f"Mermaid rendering not available: {e}")
    print("\nGraph (text):")
    print("  __start__ --> profile_data --> suggest_cleaning")
    print("  suggest_cleaning --> approval_gate")
    print("  approval_gate --(approve)--> apply_transforms --> compile_report --> __end__")
    print("  approval_gate --(reject)--> compile_report --> __end__")
    print("  approval_gate --(revise)--> suggest_cleaning")

---

# Part D — Execute the Pipeline

## 17. Run the Full Pipeline

Our simulated reviewer will **revise** on the first pass (asking for better salary handling), then **approve** on the second pass.

In [ ]:
def make_initial_state(dataframe: pd.DataFrame) -> CleaningState:
    return {
        "df_json": dataframe.to_json(orient="records"),
        "profile_report": "",
        "cleaning_plan": "",
        "decision": "",
        "approval_notes": "",
        "revision_feedback": "",
        "revision_count": 0,
        "cleaned_df_json": "",
        "transform_log": "",
        "final_report": "",
        "current_node": "start",
    }


print("Running data cleaning pipeline")
print(f"  Dataset: {df.shape[0]} rows × {df.shape[1]} columns")
print(f"  Expected flow: profile → suggest → ★revise★ → suggest → ★approve★ → apply → report")
print("=" * 70)

result = app.invoke(
    make_initial_state(df),
    {"recursion_limit": 25},
)

print(f"\nPipeline complete!")
print(f"  Decision: {result['decision']}")
print(f"  Revision rounds: {result['revision_count']}")
print(f"  Report length: {len(result['final_report'])} chars")

## 18. Inspect Intermediate State

Every output from every node is preserved in state.

In [ ]:
print("STATE INSPECTION")
print("=" * 80)

print("\n[1] PROFILE REPORT (first 500 chars):")
wrap_print(result["profile_report"][:500])

print("\n" + "-" * 80)
print("[2] CLEANING PLAN (final version after revision):")
wrap_print(result["cleaning_plan"][:600])

print("\n" + "-" * 80)
print(f"[3] DECISION: {result['decision'].upper()}")
print(f"    Notes: {result['approval_notes']}")
print(f"    Revisions: {result['revision_count']}")

print("\n" + "-" * 80)
print("[4] TRANSFORM LOG:")
wrap_print(result["transform_log"])

## 19. Step-by-Step Streaming View

Watch the revision loop in action via `app.stream()`.

In [ ]:
print("STREAMING EXECUTION (with revision loop)")
print("=" * 60)

step = 0
for chunk in app.stream(make_initial_state(df), {"recursion_limit": 25}):
    step += 1
    for node_name, node_output in chunk.items():
        keys_written = [k for k in node_output.keys() if k != "current_node"]
        # Show sizes for data fields, values for small fields
        details = []
        for k in keys_written:
            val = node_output[k]
            if isinstance(val, str) and len(val) > 50:
                details.append(f"{k}: {len(val)} chars")
            else:
                details.append(f"{k}: {val}")
        print(f"  Step {step}: {node_name:<22} → {', '.join(details)}")

print(f"\nTotal steps: {step}")
print("Notice: suggest_cleaning and approval_gate each run TWICE (revision loop).")

---

# Part E — Before / After Analysis

## 20. View the Cleaning Report

In [ ]:
print(result["final_report"])

## 21. Visual Before/After Comparison

In [ ]:
df_before = pd.read_json(result["df_json"], orient="records")
df_after = pd.read_json(result["cleaned_df_json"], orient="records")

print("BEFORE / AFTER METRICS")
print("=" * 65)

metrics = [
    ("Rows", len(df_before), len(df_after)),
    ("Columns", len(df_before.columns), len(df_after.columns)),
    ("Total nulls", df_before.isna().sum().sum(), df_after.isna().sum().sum()),
    ("Duplicate rows", df_before.duplicated().sum(), df_after.duplicated().sum()),
    ("Unique departments", df_before["department"].nunique(), df_after["department"].nunique()),
]

print(f"{'Metric':<25} {'Before':>10} {'After':>10} {'Change':>10}")
print("-" * 55)
for name, before, after in metrics:
    change = after - before
    marker = "✓" if (change < 0 and name != "Columns") or (name == "Unique departments" and after == 5) else ""
    print(f"  {name:<23} {before:>10} {after:>10} {change:>+10}  {marker}")

print(f"\nSalary dtype: {df_before['salary'].dtype} → {df_after['salary'].dtype}")

In [ ]:
# Column-level null comparison
print("COLUMN-LEVEL NULL COMPARISON")
print("=" * 55)
print(f"{'Column':<25} {'Before':>8} {'After':>8} {'Fixed':>8}")
print("-" * 49)

for col in df_before.columns:
    nb = df_before[col].isna().sum()
    na = df_after[col].isna().sum() if col in df_after.columns else 0
    fixed = nb - na
    bar = "█" * min(fixed, 20)
    print(f"  {col:<23} {nb:>8} {na:>8} {fixed:>8}  {bar}")

In [ ]:
# Show cleaned data sample
print("CLEANED DATA SAMPLE (first 10 rows):")
print("=" * 80)
df_after.head(10)

## 22. Salary Distribution: Before vs After

In [ ]:
# Text-based histogram comparison
print("SALARY DISTRIBUTION COMPARISON")
print("=" * 60)

# Before: parse numeric values only
salary_before = pd.to_numeric(df_before["salary"].apply(
    lambda x: str(x).replace("$", "").replace(",", "") if pd.notna(x) else np.nan
), errors="coerce")

salary_after = df_after["salary"]

for label, s in [("BEFORE", salary_before), ("AFTER", salary_after)]:
    valid = s.dropna()
    print(f"\n  {label}:")
    print(f"    count: {len(valid)}, mean: {valid.mean():,.0f}, std: {valid.std():,.0f}")
    print(f"    min: {valid.min():,.0f}, max: {valid.max():,.0f}")
    print(f"    Q1: {valid.quantile(0.25):,.0f}, median: {valid.median():,.0f}, Q3: {valid.quantile(0.75):,.0f}")

    # Simple text histogram
    bins = [0, 30000, 50000, 70000, 90000, 110000, 300000, 1000000]
    labels = ["0-30K", "30-50K", "50-70K", "70-90K", "90-110K", "110-300K", ">300K"]
    counts = pd.cut(valid, bins=bins, labels=labels).value_counts().sort_index()
    for lbl, cnt in counts.items():
        bar = "█" * min(cnt, 50)
        print(f"    {lbl:<10} {cnt:>4} {bar}")

---

# Part F — State Accumulation & Design Lessons

## 23. State Size at Each Stage

In [ ]:
print("STATE ACCUMULATION")
print("=" * 60)

fields = [
    ("df_json (input)", result["df_json"]),
    ("profile_report", result["profile_report"]),
    ("cleaning_plan", result["cleaning_plan"]),
    ("approval_notes", result["approval_notes"]),
    ("transform_log", result["transform_log"]),
    ("cleaned_df_json", result["cleaned_df_json"]),
    ("final_report", result["final_report"]),
]

cumulative = 0
for name, content in fields:
    size = len(content)
    cumulative += size
    bar = "█" * min(size // 200, 40)
    print(f"  {name:<22} {size:>6} chars  {bar}")

print(f"\n  Total state: {cumulative:,} chars")
print(f"  Each node only ADDS its output — never overwrites other fields.")

## 24. Node Design Recap

```
  Node                READS                        WRITES              LLM?
  ─────────────────── ──────────────────────────── ─────────────────── ─────
  profile_data        df_json                      profile_report      No
  suggest_cleaning    profile_report,              cleaning_plan       Yes
                      revision_feedback (opt.)
  approval_gate       profile_report,              decision,           No
                      cleaning_plan                approval_notes,
                                                   revision_feedback,
                                                   revision_count
  apply_transforms    df_json                      cleaned_df_json,    No
                                                   transform_log
  compile_report      df_json, cleaned_df_json,    final_report        No
                      decision, approval_notes,
                      transform_log

  NOTICE: Only ONE node (suggest_cleaning) uses the LLM.
  All data manipulation is deterministic pandas code.
  This is intentional — LLMs suggest, humans approve,
  and hard-coded transforms execute safely.
```

## 25. The Human-in-the-Loop Pattern: When and How

| Implementation | Best For | How It Works |
|----------------|----------|-------------|
| **Simulated decisions** | Testing, demos, CI/CD | Pre-defined approve/revise/reject |
| **`input()` in notebook** | Interactive EDA | Graph pauses at approval, user types response |
| **LangGraph `interrupt()`** | Production workflows | Built-in pause + resume with checkpointing |
| **Webhook callback** | Async approval flows | Post to Slack → wait for reaction → resume |
| **Database queue** | Multi-user systems | Write pending reviews to DB → poll for decisions |

```python
# Production pattern with LangGraph interrupt
from langgraph.checkpoint.memory import MemorySaver

checkpointer = MemorySaver()
app = workflow.compile(
    checkpointer=checkpointer,
    interrupt_before=["approval_gate"],  # pause BEFORE this node
)

# First run: graph runs until it hits the interrupt
config = {"configurable": {"thread_id": "cleaning-job-42"}}
state = app.invoke(initial_state, config)
# → state has profile + cleaning_plan, but decision is empty

# Human reviews externally (Slack, web UI, email, etc.)
# Then resume with the decision:
app.update_state(config, {"decision": "approve", "approval_notes": "LGTM"})
final = app.invoke(None, config)  # continues from checkpoint
```

## 26. Common Mistakes in Data Cleaning Pipelines

| Mistake | Problem | Fix |
|---------|---------|-----|
| LLM generates pandas code and you `exec()` it | Arbitrary code execution on your data | Hard-code the transform operations |
| No before/after comparison | Can't verify cleaning didn't destroy signal | Always compute and display metrics |
| Dropping rows without counting | Silently lose 30% of data | Log every row/null removal with counts |
| No revision loop | Bad LLM suggestions get applied directly | Add human review + revision capability |
| No cap on revisions | Infinite loop if reviewer keeps asking for changes | MAX_REVISIONS safety guard |
| Profile and suggest in same node | Can't audit whether stats are correct | Separate deterministic profiling from LLM reasoning |

## 27. Exercises

### Exercise 1: Interactive Approval
Replace the simulated decisions with `input()`. Let the user type "approve", "revise", or "reject" and optionally add feedback. Test with a small dataset.

### Exercise 2: Selective Approval
Modify the approval gate so the human can approve or reject **individual actions** from the cleaning plan (e.g., "approve actions 1,3,5 — reject 2,4"). Update `apply_transforms` to only execute approved actions.

### Exercise 3: Undo Capability
Add an `undo_transform` node that lets the user revert the last cleaning operation. Use the `df_json` (original) and `transform_log` to reconstruct any intermediate state.

### Mini Challenge: Multi-Dataset Pipeline
Extend the workflow to accept multiple DataFrames (e.g., a main table and a lookup table). Add a `check_referential_integrity` node that validates foreign key relationships before and after cleaning.

In [ ]:
print("SESSION SUMMARY")
print("=" * 65)
print(f"Dataset:         {len(df_before)} rows → {len(df_after)} rows")
print(f"Nulls fixed:     {df_before.isna().sum().sum()} → {df_after.isna().sum().sum()}")
print(f"Duplicates:      {df_before.duplicated().sum()} → {df_after.duplicated().sum()}")
print(f"Decision:        {result['decision']}")
print(f"Revisions:       {result['revision_count']}")
print()
print("Graph topology:")
print("  START → profile_data → suggest_cleaning → approval_gate")
print("  → approval_gate ──(approve)──→ apply_transforms → compile_report → END")
print("  → approval_gate ──(reject)───→ compile_report → END")
print("  → approval_gate ──(revise)───→ suggest_cleaning (loop)")
print()
print("Nodes built:")
print("  1. profile_data      — deterministic stats (no LLM)")
print("  2. suggest_cleaning  — LLM proposes ranked actions")
print("  3. approval_gate     — ★ human review with approve/revise/reject")
print("  4. apply_transforms  — deterministic pandas operations (no LLM)")
print("  5. compile_report    — before/after comparison + audit trail")
print()
print("Key patterns:")
print("  • Human-in-the-loop: LLM suggests, human approves, code executes")
print("  • Separation of concerns: only 1 of 5 nodes uses the LLM")
print("  • Revision loop with feedback for iterative improvement")
print("  • Full audit trail: every transform logged with counts")
print("  • No exec() of LLM output: hard-coded safe transforms only")

## 28. Key Takeaways

| # | Takeaway |
|---|----------|
| 1 | **Separate profiling from suggestion** — stats are factual, suggestions are judgmental. Keep them in different nodes |
| 2 | **Never `exec()` LLM-generated code on real data** — the LLM suggests actions, hard-coded transforms execute them |
| 3 | **Human-in-the-loop is essential for data cleaning** — dropping rows silently can destroy entire analyses |
| 4 | **Revision loops improve plans** — reviewer feedback in round 1 catches issues the LLM missed |
| 5 | **Cap revision loops** — `MAX_REVISIONS` prevents infinite cycling |
| 6 | **Before/after metrics prove cleaning worked** — compare row counts, null counts, and distributions |
| 7 | **Each node returns ONLY changed fields** — LangGraph merges partial updates into full state |
| 8 | **The transform log is your audit trail** — every action recorded with exact counts and values |
| 9 | **Streaming shows the loop** — `app.stream()` makes revision cycles visible in real time |
| 10 | **Production approval uses LangGraph checkpoints** — `interrupt_before` pauses the graph for real human review |

---

*This notebook is part of the [Machine Learning Projects](https://github.com/pypi-ahmad/Machine-Learning-Projects) repository.*